# Equations and root finding

```{admonition} Learning outcomes
After working through this chapter, you should be able to:

1. formulate an equation as a root-finding problem $f(x)=0$
2. explain and implement the bisection method and Newton's method
3. use a tolerance and a maximum number of iterations to control a numerical calculation
4. assess strengths, weaknesses and convergence of different root-finding methods
5. use ready-made root solvers in SciPy for chemical problems
```

Many chemical problems end with an equation that we need to solve. Some equations can be solved analytically, but the expressions are often complicated enough that a numerical solution is more practical.

The central idea is simple. An equation

$$g(x)=h(x)$$

can be written as

$$f(x)=g(x)-h(x)=0.$$

A **root** of a function is an $x$ value for which the function value is zero. Solving $g(x)=h(x)$ is therefore equivalent to finding a root of $f(x)=g(x)-h(x)$. We have turned the equation into a **root-finding problem**.

## A chemical example: pH of a weak acid

We use a 0.010 M solution of acetic acid as an example. For a monoprotic weak acid with total concentration $C$, the mass balance and acid dissociation constant can be combined to give the concentration of the conjugate base:

$$[\mathrm{A^-}]=C\frac{K_a}{[\mathrm{H_3O^+}]+K_a}.$$

The charge balance is

$$[\mathrm{H_3O^+}]=[\mathrm{A^-}]+[\mathrm{OH^-}],$$

and the ion product of water gives

$$[\mathrm{OH^-}]=\frac{K_w}{[\mathrm{H_3O^+}]}.$$

If we let $h=[\mathrm{H_3O^+}]$, the whole problem can be collected in one function:

$$f(h)=h-C\frac{K_a}{h+K_a}-\frac{K_w}{h}.$$

The pH is found when the charge balance is satisfied, that is, when $f(h)=0$.

```{admonition} Why is this useful?
:class: note
We do not need to isolate $h$ algebraically. If we can evaluate $f(h)$, we can search numerically for the value of $h$ that makes the function zero.
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 0.010
Ka = 1.75e-5
Kw = 1.0e-14

def charge_balance(h):
    A_minus = C * Ka / (h + Ka)
    OH = Kw / h
    return h - A_minus - OH

h = np.logspace(-7, -2, 500)
plt.semilogx(h, charge_balance(h))
plt.axhline(0)
plt.xlabel(r"$[\mathrm{H_3O^+}]$ (mol/L)")
plt.ylabel("Charge balance")
plt.show()

The graph shows approximately where the root lies. It is often useful to **visualise the problem before applying a numerical method**. A plot can reveal whether several roots are present and help us choose a sensible starting interval.

## First idea: look for a sign change

If $f(x_i)$ and $f(x_{i+1})$ have opposite signs, and the function is continuous between the points, there must be at least one root in the interval. A simple search can therefore move through a sequence of points and stop when the sign changes.

This is not the most efficient method, but the idea leads directly to the bisection method.

In [ ]:
x = np.logspace(-7, -2, 1000)
y = charge_balance(x)

for i in range(len(x) - 1):
    if y[i] * y[i + 1] < 0:
        a = x[i]
        b = x[i + 1]
        break

print("The root lies between", a, "and", b, "mol/L")
print("A first estimate is the midpoint:", (a + b)/2)

## The bisection method

The bisection method starts with an interval $[a,b]$ where $f(a)$ and $f(b)$ have opposite signs. We calculate the midpoint

$$m=\frac{a+b}{2}$$

and keep the half of the interval that still contains a sign change. The process is repeated until the interval or the function value is sufficiently small.

The method is not always the fastest, but it is very robust when the problem is continuous and we have a valid starting interval.

```{admonition} Tolerance
:class: note
In numerical calculations we normally should not test `f(x) == 0`. Instead, we stop when $|f(x)|$ is smaller than a chosen tolerance.
```

In [ ]:
def bisection(f, a, b, tol=1e-10, max_iterations=100):
    if f(a) * f(b) > 0:
        raise ValueError("f(a) and f(b) must have opposite signs.")

    for i in range(max_iterations):
        m = (a + b) / 2

        if abs(f(m)) < tol:
            return m, i + 1

        if f(a) * f(m) < 0:
            b = m
        else:
            a = m

    raise RuntimeError("The method did not converge within the maximum number of iterations.")

h_root, iterations = bisection(charge_balance, 1e-7, 1e-2)
pH = -np.log10(h_root)

print(f"[H3O+] = {h_root:.6e} mol/L")
print(f"pH = {pH:.3f}")
print("Iterations:", iterations)

### Try it yourself

Complete the bisection method in the editor and use it to find the pH of the weak acid.

<iframe src="../../basthon/?from=examples/equations_bisection.py" width="100%" height="600" frameborder="0" title="Try it yourself: the bisection method" loading="lazy" allowfullscreen></iframe>

## Newton's method

The bisection method uses only function values. Newton's method also uses the derivative.

You may know the notation $f'(x)$ from mathematics. In the next chapter we will also use the notation $\frac{df}{dx}$; both describe the derivative of $f$ with respect to $x$.

The tangent at $x_n$ can be written

$$y=f(x_n)+f'(x_n)(x-x_n).$$

If we set $y=0$ and solve for $x$, we obtain the next estimate:

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.$$

This is repeated until $f(x_n)$ is sufficiently close to zero.

In [ ]:
def newton_method(f, f_derivative, x0, tol=1e-10, max_iterations=50):
    x = x0

    for i in range(max_iterations):
        dfdx = f_derivative(x)
        if abs(dfdx) < 1e-14:
            raise RuntimeError("The derivative is too close to zero.")

        x_new = x - f(x) / dfdx

        if abs(f(x_new)) < tol:
            return x_new, i + 1

        x = x_new

    raise RuntimeError("The method did not converge within the maximum number of iterations.")

def f(x):
    return x**2 - 2

def f_derivative(x):
    return 2*x

root, iterations = newton_method(f, f_derivative, x0=1.0)
print("Root:", root)
print("Iterations:", iterations)

### When Newton does not behave nicely

Newton's method often converges very quickly, but it is more sensitive to the starting guess. It can also run into trouble if $f'(x)$ becomes zero or very small.

For the function

$$f(x)=x^3-2x+2$$

the starting guess $x_0=0$ gives $x_1=1$, while $x_1=1$ sends us back to $x_2=0$. The method therefore enters a cycle instead of finding a root.

In [ ]:
def f_problem(x):
    return x**3 - 2*x + 2

def df_problem(x):
    return 3*x**2 - 2

x = 0.0
for i in range(6):
    print(i, x)
    x = x - f_problem(x) / df_problem(x)

```{admonition} Exercise along the way
:class: tip
Try other starting values. Which starting values lead to a root, and which cause problems? What does this tell you about the difference between the bisection method and Newton's method?
```

## Ready-made solvers in SciPy

Once we understand the principle, it is common to use tested algorithms from numerical libraries. `scipy.optimize.root_scalar` provides several methods for one-dimensional root-finding problems.

In [ ]:
from scipy.optimize import root_scalar

bisect_result = root_scalar(charge_balance, bracket=[1e-7, 1e-2], method="bisect")

def d_charge_balance(h):
    return 1 + C*Ka/(h + Ka)**2 + Kw/h**2

newton_result = root_scalar(
    charge_balance, x0=4e-4, fprime=d_charge_balance, method="newton"
)

print("Bisection:")
print("  converged:", bisect_result.converged)
print("  iterations:", bisect_result.iterations)
print("  pH:", -np.log10(bisect_result.root))

print("\nNewton:")
print("  converged:", newton_result.converged)
print("  iterations:", newton_result.iterations)
print("  pH:", -np.log10(newton_result.root))

## Which method should we choose?

| Situation | A natural choice |
|---|---|
| We know an interval containing a sign change | Bisection or another bracketed method |
| We have a good starting value and know the derivative | Newton |
| We want a robust ready-made solver | `root_scalar` with an appropriate method |
| There may be several roots | Plot or scan the interval first |

The important point is not only to obtain a number, but to check that the number actually solves the chemical problem.

```{admonition} Numerical workflow
:class: important
1. Formulate the chemistry as $f(x)=0$.
2. Inspect the function and choose a sensible search interval.
3. Choose a method, tolerance and, if needed, a starting guess.
4. Check that the method converged.
5. Substitute the solution back into the model and assess whether it is chemically reasonable.
```

## Short summary

- Equations can be formulated as root-finding problems.
- The bisection method is robust when we have a sign change.
- Newton's method can be fast, but is more sensitive to the starting guess and derivative.
- A tolerance and a maximum number of iterations make the calculation controllable.
- SciPy provides ready-made solvers, but we should still understand the problem we give them.

## Exercises

```{admonition} Exercise 1 – pH of a weak acid
:class: tip
Use `bisection` to find the pH of 0.0250 M acetic acid with $K_a=1.75\cdot10^{-5}$. Compare with the approximation $[\mathrm{H_3O^+}]\approx\sqrt{K_aC}$. How large is the difference?
```

```{admonition} Exercise 2 – choose a method
:class: tip
You need to solve three problems:

1. A function has a known sign change between 2 and 3, but its derivative is difficult to calculate.
2. You know a good starting value and both $f(x)$ and $f'(x)$ are easy to calculate.
3. You suspect that the function has three roots in the interval $[-5,5]$.

Choose an approach for each case and justify your choices.
```

```{admonition} Exercise 3 – several roots
:class: tip
Find all solutions of $x^5=5x^3+3$. First plot the root function. Then use the bisection method or `root_scalar` on suitable subintervals.
```

```{admonition} Exercise 4 – Newton and the starting guess
:class: tip
Investigate $f(x)=x^3-2x+2$ with Newton's method. Test at least five different starting values. Explain why the same method can succeed from one starting value and fail from another.
```

```{admonition} Exercise 5 – chemical equilibrium
:class: tip
For the reaction $\mathrm{A \rightleftharpoons B}$, we start with 1.00 M A and 0 M B. At equilibrium, $[B]=x$ and $[A]=1-x$. Let $K=3.5$ and formulate $K=[B]/[A]$ as a root-finding problem. Find $x$ numerically and check the solution analytically.
```

```{admonition} Exercise 6 – temperature at which a process changes spontaneity
:class: tip
Assume that $\Delta H=45.0$ kJ/mol and $\Delta S=125$ J/(mol K) are constant over a temperature interval. Formulate $\Delta G(T)=\Delta H-T\Delta S=0$ as a root-finding problem and find the temperature. This equation is easy to solve analytically, so use that result to check the numerical method.
```